# Практическая работа №1
## Случайный лес: классификация сортов фасоли

**Цель работы** — применение случайного леса для многоклассовой классификации сортов фасоли по геометрическим признакам.

В работе был использован набор данных Dry Bean, содержащий семь классов. Были выполнены проверка и подготовка данных, анализ распределений и взаимосвязей признаков. После обучения случайного леса качество классификации было оценено по Accuracy, Macro Precision, Macro Recall и Macro F1-score. Также были рассмотрены ошибки по отдельным классам и важность признаков.

In [ ]:
%pip install -q pandas numpy matplotlib seaborn scipy scikit-learn

Для работы были выбраны библиотеки `pandas`, `numpy`, `matplotlib`, `seaborn`, `scipy` и `scikit-learn`. Команда `%pip install` устанавливает их в среду, в которой открыт ноутбук. Параметр `-q` уменьшает количество служебных сообщений при установке.

После запуска ячейки могут появиться сообщения об установке пакетов. Числовых результатов анализа на данном этапе ещё нет: библиотеки только подготавливаются к дальнейшей работе.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    RocCurveDisplay,
)

# В браузерной среде используется open_url, в локальной — pandas.
try:
    from pyodide.http import open_url
except ImportError:
    open_url = None

from scipy import stats
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import (precision_recall_curve, average_precision_score, roc_curve)
from sklearn.calibration import calibration_curve


Были подключены инструменты для обработки таблиц, вычислений, визуализации и обучения моделей. Для работы с файлами используется `Path`, таблицами — `pandas`, массивами — `numpy`. Графики строятся средствами `matplotlib` и `seaborn`. Из `scikit-learn` импортированы алгоритмы, функции разделения выборки, преобразования признаков и метрики качества. `scipy.stats` применяется для статистических графиков.

Отдельный блок `try / except ImportError` проверяет возможность загрузки файлов через `pyodide.http.open_url`. Это сделано для совместимости с браузерным Jupyter. В обычном Python переменная `open_url` остаётся равной `None`. Если все импорты выполнены без ошибок, ячейка ничего не печатает, но все указанные функции становятся доступны далее.

In [ ]:
def read_csv_compat(url, local_path=None, **read_csv_kwargs):
    if local_path is not None:
        local_path = Path(local_path)
        if local_path.exists():
            return pd.read_csv(local_path, **read_csv_kwargs)

    if open_url is not None:
        data = pd.read_csv(open_url(url), **read_csv_kwargs)
    else:
        data = pd.read_csv(url, **read_csv_kwargs)

    if local_path is not None:
        local_path.parent.mkdir(parents=True, exist_ok=True)
        data.to_csv(local_path, index=False)

    return data


Была задана функция `read_csv_compat`, которая используется при загрузке всех четырёх наборов данных. Сначала проверяется наличие локального CSV. Если он уже сохранён, таблица читается с диска. При отсутствии файла выполняется загрузка по URL: в браузерной среде через `open_url`, в обычном Jupyter через `pd.read_csv`. После загрузки данные записываются в папку `data`.

Аргумент `**read_csv_kwargs` позволяет при необходимости передавать дополнительные настройки чтения CSV. Команда `return data` возвращает подготовленную таблицу `DataFrame`. На экране после объявления функции ничего не появляется: фактическое чтение будет выполнено при её вызове.

## 1. Загрузка данных

In [ ]:
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

data_path = DATA_DIR / "Dry_Bean_Dataset.csv"
data_url = "https://raw.githubusercontent.com/Apress/reinforcement-learning-finance/main/Dry_Bean_Dataset.csv"

df = read_csv_compat(data_url, data_path)
df.head()


Был указан адрес исходного CSV и путь к локальной копии. Выражение `DATA_DIR.mkdir(exist_ok=True)` создаёт папку `data`, если её ещё нет. Функция `read_csv_compat` возвращает таблицу в переменную `df`. Завершающая команда `df.head()` показывает пять первых строк.

В выводе отображаются названия столбцов и значения первых пяти объектов. По этой таблице проверяется правильность загрузки: не съехали ли столбцы, как представлены признаки и где находится целевая переменная. Это первичный осмотр данных, а не результат работы алгоритма.

## 2. Первичный анализ

In [ ]:
print("Размер датасета:", df.shape)
display(df.head())
df.info()

Для первичной проверки были использованы `df.shape`, `df.head()` и `df.info()`. Первая команда показывает размер набора данных в виде `(число строк, число столбцов)`. Вторая выводит небольшой фрагмент таблицы. Третья показывает тип данных каждого столбца и количество непустых значений (`Non-Null Count`).

Если для какого-либо признака число непустых значений меньше общего числа строк, в нём имеются пропуски. Также по `dtype` можно проверить, что числовые признаки действительно распознаны как числа, а названия категорий — как текст.

In [ ]:
df.describe(include='all').T

Была получена описательная статистика с помощью `describe()`. Показатель `count` отражает число заполненных значений; `mean` — среднее; `std` — стандартное отклонение; `min` и `max` — границы наблюдаемого диапазона. Квартили `25%`, `50%` и `75%` характеризуют положение основной части данных, при этом `50%` соответствует медиане. Операция `.T` разворачивает таблицу, чтобы признаки располагались строками.

По полученной таблице удобно сравнивать диапазоны значений. Большая разница между средним и медианой может говорить об асимметрии распределения, а крайние минимумы и максимумы требуют дополнительной проверки.

## 3. Проверка качества данных

In [ ]:
print("Пропуски:")
display(df.isnull().sum().to_frame("missing"))

print("Количество полных дубликатов:", df.duplicated().sum())
print("\nРаспределение классов:")
display(df["Class"].value_counts())

Проверка пропусков была выполнена выражением `df.isnull().sum()`: оно подсчитывает отсутствующие значения для каждого столбца. Отдельно командой `df.duplicated().sum()` определяется число полностью совпадающих строк. Таблица `missing` содержит количество пропусков, а следующая строка вывода — число дубликатов.

Нулевые значения в этих результатах означают, что соответствующая проблема не обнаружена. Ненулевые значения показывают, какие столбцы или строки требуют обработки. Для задач классификации дополнительно выводится количество объектов каждого класса: это позволяет оценить их представленность ещё до обучения.

In [ ]:
feature_columns = [column for column in df.columns if column != "Class"]

outliers = {}
for column in feature_columns:
    values = df[column].dropna()
    q1 = values.quantile(0.25)
    q3 = values.quantile(0.75)
    iqr = q3 - q1
    low = q1 - 1.5 * iqr
    high = q3 + 1.5 * iqr
    outliers[column] = ((values < low) | (values > high)).sum()

pd.Series(outliers, name="Количество значений за границами IQR").sort_values(ascending=False)

По шестнадцати числовым признакам фасоли был проведён поиск потенциальных выбросов методом IQR. Список признаков сформирован из всех столбцов, кроме целевого `Class`. Для каждого признака вычисляются квартильные границы и подсчитывается число значений вне интервала `Q1 - 1.5 × IQR` — `Q3 + 1.5 × IQR`.

Итоговая таблица показывает, в каких характеристиках встречается больше необычных наблюдений. Они не считаются ошибочными автоматически, поскольку размеры и форма зерна могут заметно различаться между сортами.

In [ ]:
fig, axes = plt.subplots(4, 4, figsize=(16, 13))
axes = axes.ravel()

for index, column in enumerate(feature_columns):
    axes[index].boxplot(df[column].dropna())
    axes[index].set_title(column, fontsize=9)

for index in range(len(feature_columns), len(axes)):
    axes[index].axis("off")

plt.suptitle("Диаграммы размаха признаков Dry Bean", y=1.01)
plt.tight_layout()
plt.show()

Для каждого числового признака фасоли показан отдельный boxplot с собственной шкалой.

Так удобнее искать потенциальные аномалии в признаках с сильно различающимися диапазонами. Точки за «усами» рассматриваются как кандидаты на выбросы, но не удалялись автоматически.

In [ ]:
before = len(df)
df = df.drop_duplicates().dropna().copy()

print("Строк до очистки:", before)
print("Строк после очистки:", len(df))

Были удалены полностью совпадающие строки и записи с пропусками: использованы `drop_duplicates()` и `dropna()`. Команда `copy()` создаёт отдельную таблицу после обработки.

В результате печатаются числа строк до и после очистки. Изменение размера показывает количество исключённых объектов. Дальнейшие расчёты выполняются на оставшихся данных.

## 4. Разведочный анализ

In [ ]:
df[feature_columns].hist(figsize=(16, 14), bins=25)
plt.suptitle("Распределения числовых признаков", y=1.01)
plt.tight_layout()
plt.show()

Для каждого геометрического параметра фасоли построена гистограмма распределения.

Можно увидеть диапазоны значений, перекосы и несколько групп наблюдений. Это особенно полезно при большом количестве признаков.

In [ ]:
plt.figure(figsize=(10, 5))
order = df["Class"].value_counts().index
sns.countplot(data=df, x="Class", order=order, color=sns.color_palette()[0])
plt.title("Количество объектов каждого сорта")
plt.xticks(rotation=45)
plt.show()

Столбцы показывают количество зерен каждого из семи сортов фасоли.

Классы представлены неодинаково, поэтому кроме общей Accuracy используются Macro Precision, Macro Recall и Macro F1: при таком усреднении каждый сорт получает одинаковый вес.

In [ ]:
plt.figure(figsize=(14, 11))
sns.heatmap(
    df[feature_columns].corr(),
    cmap="coolwarm",
    center=0
)
plt.title("Матрица корреляции признаков")
plt.show()

Матрица показывает корреляцию между 16 геометрическими признаками.

Позволяет обнаружить признаки, которые сильно дублируют друг друга по информации. Random Forest может работать и с коррелирующими признаками, но эта визуализация помогает понять структуру датасета.

### Scatter plot: площадь и периметр фасоли по сортам

In [ ]:
plot_sample = df.sample(n=min(2500, len(df)), random_state=42)
plt.figure(figsize=(9, 6))
sns.scatterplot(data=plot_sample, x="Area", y="Perimeter", hue="Class", s=19, alpha=0.5)
plt.xlabel("Площадь зерна, пиксели²")
plt.ylabel("Периметр зерна, пиксели")
plt.title("Scatter plot: площадь и периметр фасоли по сортам")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left", title="Сорт")
plt.tight_layout()
plt.show()

Была построена диаграмма рассеяния с раскраской по семи классам. Для читаемости берётся воспроизводимая случайная подвыборка до 2500 зерен; `hue="Class"` задаёт цвет реального сорта.

X — площадь в пикселях², Y — периметр в пикселях, цвет — сорт. Наложение точек показывает области, в которых классы похожи по двум признакам.

Наблюдаемая двухмерная разделимость **не равна** точности случайного леса: он использует все 16 признаков одновременно.

### Violin plot: распределение компактности по сортам

In [ ]:
plt.figure(figsize=(11, 5))
sns.violinplot(data=df, x="Class", y="Compactness", inner="quartile", cut=0, color=sns.color_palette()[0])
plt.xlabel("Сорт фасоли")
plt.ylabel("Компактность (безразмерная)")
plt.title("Violin plot: распределение компактности по сортам")
plt.xticks(rotation=35)
plt.tight_layout()
plt.show()

Была построена скрипичная диаграмма (*violin plot*). Каждая скрипка представляет оценку плотности `Compactness` в одном из семи сортов, линии внутри — квартили.

X — сорт, Y — компактность, ширина скрипки — локальная плотность наблюдений.

Если распределения значительно различаются, компактность потенциально информативна для классификации. Перекрытие распределений означает, что по одной компактности определение сорта неоднозначно.

### Групповой boxplot: площадь зерен по сортам

In [ ]:
plt.figure(figsize=(11, 5))
sns.boxplot(data=df, x="Class", y="Area", color=sns.color_palette()[0])
plt.xlabel("Сорт фасоли")
plt.ylabel("Площадь зерна, пиксели²")
plt.title("Групповой boxplot: площадь зерен по сортам")
plt.xticks(rotation=35)
plt.tight_layout()
plt.show()

Были построены групповые диаграммы размаха. `sns.boxplot` строит распределение площади для каждого сорта.

X — названия семи сортов, Y — площадь зерна; внутри каждой коробки медиана и межквартильный размах, за «усами» — возможные выбросы.

Можно сопоставить типичные размеры зерен и разброс в классах. Большая коробка означает высокий вариационный размах, а не обязательно плохие данные.

### ECDF округлости для трёх наиболее представленных сортов

In [ ]:
top3_classes = df["Class"].value_counts().head(3).index.tolist()
plt.figure(figsize=(9, 5))
for cls in top3_classes:
    values = np.sort(df.loc[df["Class"] == cls, "roundness"].to_numpy())
    share = np.arange(1, len(values)+1) / len(values)
    plt.step(values, share, where="post", label=cls)
plt.xlabel("Округлость зерна (roundness, безразмерная)")
plt.ylabel("Доля зерен с округлостью ≤ X")
plt.title("ECDF округлости для трёх наиболее представленных сортов")
plt.ylim(0, 1.02)
plt.legend()
plt.tight_layout()
plt.show()

Были построены эмпирические функции распределения (*ECDF*) по трём наиболее многочисленным сортам. `value_counts().head(3)` выбирает 3 класса, `np.sort` и накопленные доли строят отдельные кривые.

По X — безразмерная округлость, по Y — доля объектов с округлостью не выше выбранного значения.

Разнесённые линии указывают на различия распределений округлости. Показаны только три класса для читаемости; **это не сравнительная оценка всех семи сортов**.

## 5. Подготовка данных

In [ ]:
encoder = LabelEncoder()

X = df[feature_columns]
y = encoder.fit_transform(df["Class"])

print("Классы:")
print(list(encoder.classes_))

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train:", X_train.shape)
print("Test:", X_test.shape)

Числовые характеристики фасоли были выделены в `X`, а названия сортов из столбца `Class` закодированы через `LabelEncoder` в `y`. Вывод `encoder.classes_` показывает порядок семи сортов и позволяет восстановить соответствие между числовой меткой и названием.

Затем выборка была разделена на 80% Train и 20% Test. Параметр `stratify=y` сохраняет примерное соотношение сортов в двух частях. Напечатанные размеры отражают число объектов и шестнадцать входных признаков.

## 6. Обучение

In [ ]:
model = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    min_samples_leaf=1,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print(f"Accuracy на train: {model.score(X_train, y_train):.3f}")
print(f"Accuracy на test:  {model.score(X_test, y_test):.3f}")

Был обучен случайный лес `RandomForestClassifier`, состоящий из 200 деревьев (`n_estimators=200`). Глубина отдельных деревьев не ограничена параметром `max_depth=None`, а `min_samples_leaf=1` допускает лист с одним объектом. `n_jobs=-1` разрешает использовать доступные процессорные ядра, `random_state=42` фиксирует случайность. При `fit()` обучается весь ансамбль, а `predict()` возвращает итоговый класс по голосованию деревьев.

В выводе показываются Accuracy на Train и Test. При оценке модели важнее значение на тесте; заметный разрыв между обучением и тестом может свидетельствовать о подгонке под тренировочную выборку.

## 7. Метрики

In [ ]:
accuracy = accuracy_score(y_test, y_pred)
precision_macro = precision_score(y_test, y_pred, average="macro")
recall_macro = recall_score(y_test, y_pred, average="macro")
f1_macro = f1_score(y_test, y_pred, average="macro")

print(f"Accuracy:        {accuracy:.3f}")
print(f"Macro Precision: {precision_macro:.3f}")
print(f"Macro Recall:    {recall_macro:.3f}")
print(f"Macro F1-score:  {f1_macro:.3f}")
print()
print(
    classification_report(
        y_test,
        y_pred,
        target_names=encoder.classes_,
        digits=3
    )
)

Для семи классов рассчитаны Accuracy, Macro Precision, Macro Recall и Macro F1. При `average="macro"` показатель сначала определяется отдельно для каждого сорта, а затем значения усредняются с одинаковым весом. Поэтому редкие классы не теряются в общей оценке.

В выводе находятся четыре итоговых числа и `classification_report` с показателями по каждому сорту. Если Macro F1 заметно ниже Accuracy, это повод проверить, какие именно классы распознаются хуже.

In [ ]:
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(9, 7))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=encoder.classes_,
    yticklabels=encoder.classes_
)
plt.xlabel("Предсказанный класс")
plt.ylabel("Фактический класс")
plt.title("Матрица ошибок")
plt.xticks(rotation=45)
plt.yticks(rotation=0)
plt.show()

Это матрица ошибок для семи классов. Строки соответствуют фактическим сортам, столбцы — предсказанным.

Главная диагональ содержит правильные ответы. Значения вне диагонали показывают, какие конкретно сорта модель путает между собой.

Для матрицы ошибок особенно важно не только общее число правильных ответов, но и те пары сортов, между которыми возникают систематические ошибки.

## 8. Важность признаков

In [ ]:
importance = pd.DataFrame({
    "Признак": feature_columns,
    "Важность": model.feature_importances_
}).sort_values("Важность", ascending=False)

display(importance)

plt.figure(figsize=(9, 6))
sns.barplot(
    data=importance.head(10),
    x="Важность",
    y="Признак",
    color=sns.color_palette()[0]
)
plt.title("10 наиболее важных признаков")
plt.show()

Показаны десять признаков с наибольшей важностью по оценке случайного леса.

Чем длиннее столбец, тем больший вклад признак в среднем вносил в полезные разделения всех деревьев леса. В отличие от одиночного дерева, здесь важность агрегируется по всему ансамблю.

### Нормированная матрица ошибок леса (по строкам)

In [ ]:
cm_norm = confusion_matrix(y_test, y_pred, normalize="true")
plt.figure(figsize=(10, 7))
sns.heatmap(cm_norm, annot=True, fmt=".1%", cmap="Blues", vmin=0, vmax=1,
            xticklabels=encoder.classes_, yticklabels=encoder.classes_)
plt.xlabel("Предсказанный сорт")
plt.ylabel("Фактический сорт")
plt.title("Нормированная матрица ошибок леса (по строкам)")
plt.xticks(rotation=40)
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

Была построена матрица ошибок с нормировкой по фактическому классу. `normalize="true"` делит значения каждой строки на число тестовых объектов реального сорта.

Каждая строка суммируется до 100%; диагональ соответствует полноте распознавания сорта, другие клетки — видам ошибок.

Проценты удобнее численностей для сравнения качества распознавания классов разного размера.

### Сравнение Precision, Recall и F1 для семи сортов

In [ ]:
class_metrics = classification_report(y_test, y_pred, target_names=encoder.classes_,
                                      output_dict=True, zero_division=0)
pos = np.arange(len(encoder.classes_))
w = 0.26
plt.figure(figsize=(12, 5))
for j, metric in enumerate(["precision", "recall", "f1-score"]):
    vals = [class_metrics[c][metric] for c in encoder.classes_]
    plt.bar(pos + (j-1)*w, vals, w, label=metric)
plt.xticks(pos, encoder.classes_, rotation=35)
plt.ylabel("Значение метрики [0, 1]")
plt.ylim(0, 1.05)
plt.title("Сравнение Precision, Recall и F1 для семи сортов")
plt.legend()
plt.tight_layout()
plt.show()

Были построены сгруппированные столбчатые диаграммы качества по классам. Из `classification_report(..., output_dict=True)` извлекаются Precision/Recall/F1 каждого из семи сортов.

X — класс, Y — метрика 0–1; три рядом стоящих столбца описывают один сорт.

Значительные различия Recall/F1 между сортами показывают, какие классы сложнее для леса, даже если общая Accuracy высокая.

### Распределение уверенности: правильные и ошибочные ответы

In [ ]:
probs_rf = model.predict_proba(X_test)
confidence = probs_rf.max(axis=1)
correct_flag = y_pred == y_test
plt.figure(figsize=(9, 5))
plt.hist(confidence[correct_flag], bins=np.linspace(0, 1, 16), alpha=0.6,
         label="Верные прогнозы")
plt.hist(confidence[~correct_flag], bins=np.linspace(0, 1, 16), alpha=0.6,
         label="Ошибочные прогнозы")
plt.xlabel("Максимальная предсказанная вероятность класса")
plt.ylabel("Число тестовых объектов")
plt.title("Распределение уверенности: правильные и ошибочные ответы")
plt.legend()
plt.tight_layout()
plt.show()

Были построены наложенные гистограммы уверенности (*confidence histograms*). `predict_proba` возвращает вероятности семи классов; максимум по строке — уверенность в выбранном классе, `correct_flag` отделяет верные прогнозы от ошибок.

X — максимальная вероятность в диапазоне 0–1, Y — число тестовых зерен. Наложение показывает, бывают ли ошибки при высокой уверенности.

Высокая уверенность не гарантирует правильность. У дерева ансамбля вероятности **не обязательно калиброваны**, поэтому их нельзя автоматически считать реальными частотами событий.

### Сравнение реального и предсказанного распределений классов

In [ ]:
actual_shares = pd.Series(y_test).value_counts(normalize=True).reindex(range(len(encoder.classes_)), fill_value=0)
pred_shares = pd.Series(y_pred).value_counts(normalize=True).reindex(range(len(encoder.classes_)), fill_value=0)
pos = np.arange(len(encoder.classes_))
plt.figure(figsize=(11, 5))
plt.bar(pos - 0.2, actual_shares.to_numpy(), width=0.4, label="Фактическая доля")
plt.bar(pos + 0.2, pred_shares.to_numpy(), width=0.4, label="Предсказанная доля")
plt.xticks(pos, encoder.classes_, rotation=35)
plt.ylabel("Доля объектов тестовой выборки")
plt.title("Сравнение реального и предсказанного распределений классов")
plt.legend()
plt.tight_layout()
plt.show()

Для сравнения использован график: сгруппированная диаграмма относительных частот. `value_counts(normalize=True)` получает доли объектов по каждому классу отдельно для реальных и предсказанных меток.

X — сорт, Y — доля тестовых зерен данного сорта; два столбца в каждой группе сравнивают реальное и прогнозируемое распределение.

Сильное отличие высот показывает систематическое пере- или недопредсказание какого-либо класса. **Совпадение суммарных долей само по себе не гарантирует правильной классификации отдельных объектов.**

### Доля ошибок по каждому фактическому классу

In [ ]:
recalls = np.diag(cm_norm)
error_rates = 1 - recalls
plt.figure(figsize=(10, 5))
plt.bar(encoder.classes_, error_rates * 100)
plt.ylabel("Доля ошибочных предсказаний внутри класса, %")
plt.xlabel("Фактический сорт фасоли")
plt.title("Доля ошибок по каждому фактическому классу")
plt.xticks(rotation=35)
plt.tight_layout()
plt.show()

Была построена столбчатая диаграмма частоты ошибок по классам. Диагональ нормированной матрицы `cm_norm` содержит Recall сорта, поэтому `1 - Recall` даёт долю объектов, классифицированных неверно.

X — фактический сорт, Y — процент ошибок внутри этого сорта. Низкий столбец соответствует лучшему распознаванию сорта.

Это удобный способ выделить самые проблемные классы. Частота ошибок не отражает сама по себе цену или серьёзность разных типов ошибок.

### Накопленная важность признаков случайного леса

In [ ]:
ordered_imp = np.sort(model.feature_importances_)[::-1]
cumulative_imp = np.cumsum(ordered_imp)
plt.figure(figsize=(9, 5))
plt.plot(np.arange(1, len(cumulative_imp)+1), cumulative_imp, marker="o")
plt.axhline(0.8, ls="--", color="gray", label="80% суммарной importance")
plt.xlabel("Число наиболее важных признаков")
plt.ylabel("Накопленная доля важности (Gini importance)")
plt.title("Накопленная важность признаков случайного леса")
plt.ylim(0, 1.03)
plt.grid(alpha=0.25)
plt.legend()
plt.tight_layout()
plt.show()

Была построена кривая накопленной важности признаков (*cumulative feature importance curve*). Сортируем `feature_importances_` по убыванию и накапливаем сумму `cumsum`.

X — сколько самых важных признаков учтено, Y — какая доля общей impurity-based importance на них приходится.

Быстрый рост в начале означает, что небольшое число признаков отвечает за большую часть снижений неоднородности в лесу. Это **не доля объяснённой дисперсии и не гарантированное качество модели при удалении остальных признаков**.

## Вывод

Случайный лес был использован для многоклассовой классификации семи сортов фасоли. Для такой задачи Macro-метрики удобны тем, что каждый класс получает одинаковый вес при усреднении. Матрица ошибок показывает, какие сорта модель чаще путает между собой. Важности признаков позволяют увидеть, какие геометрические характеристики сильнее всего влияют на решения ансамбля деревьев.